# <center>**Split Data for Model Training**</center>  

---
**Purpose:** This notebook is dedicated to loading the fully engineered molecular feature dataset and splitting it into distinct training, validation, and test sets. This rigorous data partitioning ensures that machine learning models can be trained, tuned, and evaluated independently on the exact same sets of data, providing an unbiased assessment of their performances.

---

## 1. Setup Notebook
This section initializes the notebook environment by importing necessary libraries and configuring system settings relevant to data loading and splitting.

### 1.1. Configure Environment
This sub-section sets environment variables to optimize CPU core usage for numerical computations.

In [16]:
# General CPU Usage Optimization
import os
os.environ['OMP_NUM_THREADS'] = '16'
os.environ['MKL_NUM_THREADS'] = '16'
os.environ['OPENBLAS_NUM_THREADS'] = '16'
os.environ['NUMEXPR_NUM_THREADS'] = '16'

### 1.2. Import Libraries
All required Python libraries for data manipulation and data splitting are imported here.

In [26]:
# Standard Library Imports
from datetime import datetime
from pathlib import Path
import subprocess  # For getting Git commit ID

# Core Data Science Libraries
import pandas as pd

# ML Data Splitting Libraries
from sklearn.model_selection import train_test_split

### 1.3. Set Data Splits Save Location

In [18]:
splits_dir = Path("../data/splits")
splits_dir.mkdir(parents=True, exist_ok=True)
print(f"The data splits will be saved in: {splits_dir}")

The data splits will be saved in: ..\data\splits


## 2. Load Fully Feature-Engineered Data
This section loads the comprehensive dataset containing all engineered molecular features and the target variable (`pGI50`), prepared in the previous notebook.

In [29]:
features_dir = Path("../data/features")
try:
    features_df = pd.read_parquet(features_dir / "gi50_features.parquet")
    print(f"Loaded fully feature-engineered data from '{features_dir / 'gi50_features.parquet'}.")
    print(f"Shape of loaded data: {features_df.shape}")
    display(features_df.head())
    display(features_df.info())

    # Check the feature engineering metadata commit hash
    fe_metadata_filename = features_dir / 'feature_engineering_metadata.parquet'
    fe_metadata_df = pd.read_parquet(fe_metadata_filename)
    fe_commit_hash = fe_metadata_df['creation_commit_hash'].iloc[0]
    fe_timestamp = fe_metadata_df['creation_timestamp'].iloc[0]

    print("\n--- Feature Engineering Traceability ---")
    print(f"Data Origin (FE Commit Hash): {fe_commit_hash}")
    print(f"Data Creation Timestamp:      {fe_timestamp}")
    print("--------------------------------------")

except FileNotFoundError:
    print(f"Error: 'gi50_features.parquet' not found in '{features_dir}'.")
    print("Please ensure you have run the dataset-saving step in the '01_Engineer_Features.ipynb' file.")

Loaded fully feature-engineered data from '..\data\features\gi50_features.parquet.
Shape of loaded data: (18778, 2196)


,molregno,pGI50,canonical_smiles,num_activities,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,MaxPartialCharge,...,morgan_fp_2038,morgan_fp_2039,morgan_fp_2040,morgan_fp_2041,morgan_fp_2042,morgan_fp_2043,morgan_fp_2044,morgan_fp_2045,morgan_fp_2046,morgan_fp_2047
0,148,7.999957,O=c1oc2c(O)c(O)cc3c(=O)oc4c(O)c(O)cc1c4c23,1,0.005972,-0.940881,0.216285,11.818182,302.194,0.344106,...,0,0,0,0,0,0,0,0,0,0
1,666,4.823909,Cc1c(C)c2c(c(C)c1O)CCC(C)(COc1ccc(CC3SC(=O)NC3...,1,0.229569,-0.467042,0.716604,22.645161,441.549,0.285946,...,0,0,0,0,0,0,0,0,0,0
2,696,5.421428,Cc1cc(O)nc2c3c(ccc12)OC(C)(C)C=C3,7,0.047809,-0.299061,0.767926,16.388889,241.290,0.211088,...,0,0,0,0,0,0,0,0,0,0
3,717,5.583359,CC(O)(CS(=O)(=O)c1ccc(F)cc1)C(=O)Nc1ccc(C#N)c(...,1,0.342865,-4.860872,0.560412,13.965517,430.379,0.417261,...,0,0,0,0,0,1,0,0,0,0
4,846,5.405822,CC1(C)CC(C)(C)c2cc(NC(=S)Nc3ccc([N+](=O)[O-])c...,9,0.052565,-0.422109,0.375924,16.888889,401.557,0.269075,...,0,0,0,0,0,0,0,0,0,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18778 entries, 0 to 18777
Columns: 2196 entries, molregno to morgan_fp_2047
dtypes: float64(145), int64(2050), object(1)
memory usage: 314.6+ MB


None


--- Feature Engineering Traceability ---
Data Origin (FE Commit Hash): 69eb8c3
Data Creation Timestamp:      2025-10-21T02:02:48.932017
--------------------------------------


## 3. Define Features and Target
This section explicitly separates the dataset into features (X) and the target variable (y), preparing them for the splitting process.

In [20]:
X = features_df.drop(columns=['pGI50'], errors='ignore')
y = features_df['pGI50']

print(f"\nDataFrame for splitting created.")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
display(X.head())


DataFrame for splitting created.
X shape: (18778, 2195)
y shape: (18778,)


,molregno,canonical_smiles,num_activities,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,MaxPartialCharge,MinPartialCharge,...,morgan_fp_2038,morgan_fp_2039,morgan_fp_2040,morgan_fp_2041,morgan_fp_2042,morgan_fp_2043,morgan_fp_2044,morgan_fp_2045,morgan_fp_2046,morgan_fp_2047
0,148,O=c1oc2c(O)c(O)cc3c(=O)oc4c(O)c(O)cc1c4c23,1,0.005972,-0.940881,0.216285,11.818182,302.194,0.344106,-0.504143,...,0,0,0,0,0,0,0,0,0,0
1,666,Cc1c(C)c2c(c(C)c1O)CCC(C)(COc1ccc(CC3SC(=O)NC3...,1,0.229569,-0.467042,0.716604,22.645161,441.549,0.285946,-0.507381,...,0,0,0,0,0,0,0,0,0,0
2,696,Cc1cc(O)nc2c3c(ccc12)OC(C)(C)C=C3,7,0.047809,-0.299061,0.767926,16.388889,241.290,0.211088,-0.493247,...,0,0,0,0,0,0,0,0,0,0
3,717,CC(O)(CS(=O)(=O)c1ccc(F)cc1)C(=O)Nc1ccc(C#N)c(...,1,0.342865,-4.860872,0.560412,13.965517,430.379,0.417261,-0.379205,...,0,0,0,0,0,1,0,0,0,0
4,846,CC1(C)CC(C)(C)c2cc(NC(=S)Nc3ccc([N+](=O)[O-])c...,9,0.052565,-0.422109,0.375924,16.888889,401.557,0.269075,-0.332481,...,0,0,0,0,0,0,0,0,0,0


## 4. Split Data
The dataset is partitioned into training, validation, and test sets to ensure robust model development and unbiased performance evaluation.

In [21]:
# Define a consistent random state for reproducibility
RANDOM_STATE = 42

print("\nPerforming initial train-test split (85% Train+Val, 15% Test)...")
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y,
                                                            test_size=0.15, # 15% for the final test set
                                                            random_state=RANDOM_STATE,
                                                            shuffle=True)

print(f"X_train_val shape: {X_train_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train_val shape: {y_train_val.shape}")
print(f"y_test shape: {y_test.shape}")

print("\nPerforming second split (Train and Validation from Train+Val set)...")
val_size_ratio_from_train_val = 0.15 / (1 - 0.15)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val,
                                                  test_size=val_size_ratio_from_train_val,
                                                  random_state=RANDOM_STATE,
                                                  shuffle=True)

print(f"\nFinal Split Shapes:")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

# Double check total rows
total_rows_after_split = X_train.shape[0] + X_val.shape[0] + X_test.shape[0]
print(f"Total rows after splitting: {total_rows_after_split} (Should match original {X.shape[0]})")


Performing initial train-test split (85% Train+Val, 15% Test)...
X_train_val shape: (15961, 2195)
X_test shape: (2817, 2195)
y_train_val shape: (15961,)
y_test shape: (2817,)

Performing second split (Train and Validation from Train+Val set)...

Final Split Shapes:
X_train shape: (13144, 2195)
y_train shape: (13144,)
X_val shape: (2817, 2195)
y_val shape: (2817,)
X_test shape: (2817, 2195)
y_test shape: (2817,)
Total rows after splitting: 18778 (Should match original 18778)


## 5. Save Data Splits
The distinct training, validation, and test sets (both features and targets) are saved locally for direct use in subsequent model training and evaluation notebooks.

### 5.1. Save Final Data Splits

In [22]:
print(f"\nSaving training, validation, and testing splits to {splits_dir}...")

X_train.to_parquet(splits_dir / "X_train_refined.parquet", index=False)
X_val.to_parquet(splits_dir / "X_val_refined.parquet", index=False)
X_test.to_parquet(splits_dir / "X_test_refined.parquet", index=False)

y_train.to_frame().to_parquet(splits_dir / "y_train_refined.parquet", index=True)
y_val.to_frame().to_parquet(splits_dir / "y_val_refined.parquet", index=True)
y_test.to_frame().to_parquet(splits_dir / "y_test_refined.parquet", index=True)

print("Data splits saved successfully to .parquet files.")
print("\nData splitting complete. Splits are ready for all model development notebooks.")


Saving training, validation, and testing splits to ..\data\splits...
Data splits saved successfully to .parquet files.

Data splitting complete. Splits are ready for all model development notebooks.


### 5.2. Save Essential Metadata

#### 5.2.1. Get Current Git Commit ID
The current Git commit ID (hash) is programmatically retrieved. This commit ID will be saved as essential metadata to ensure direct traceability and reproducibility.

In [30]:
def get_git_commit_hash():
    try:
        # Get the short commit hash
        commit_hash = (
            subprocess.check_output(["git", "rev-parse", "--short", "HEAD"])
            .strip()
            .decode("ascii")
        )
        return commit_hash
    except (subprocess.CalledProcessError, FileNotFoundError):
        return "unknown_commit"


current_commit = get_git_commit_hash()
print(f"Current Git Commit ID: {current_commit}")

Current Git Commit ID: 14773e4


#### 5.2.2. Create and Save Metadata Dictionary

In [31]:
metadata_filename = splits_dir / "data_splitting_metadata.parquet"

metadata_dict = {
    "creation_commit_hash": current_commit,
    "creation_timestamp": datetime.now().isoformat(),
    "data_source_description": "Data splits that contain refined features after checking extreme cardinality and multicollinearity in RDKit descriptors.",
}

metadata_df = pd.DataFrame([metadata_dict])

print(f"Saving data splitting metadata dictionary to: {metadata_filename}...")
try:
    metadata_df.to_parquet(metadata_filename, index=False)
    print("Data splitting metadata saved successfully!")
except Exception as e:
    print(f"Error saving metadata: {e}")

print("\nData splitting complete.")

Saving data splitting metadata dictionary to: ..\data\splits\data_splitting_metadata.parquet...
Data splitting metadata saved successfully!

Data splitting complete.
